# 03 · Observabilidad — Consultas sobre Event Logs del Pipeline

Fuente: `fintech_finpay.observability` (event logs habilitados en el pipeline DLT).

Estas queries alimentan el **Dashboard AI/BI de Observabilidad**.

In [ ]:
USE CATALOG fintech_finpay;

## 1 · Registros procesados exitosamente por capa

In [ ]:
SELECT
  CASE
    WHEN origin.flow_name LIKE '%.bronze.%' THEN 'Bronze'
    WHEN origin.flow_name LIKE '%.silver.%' THEN 'Silver'
    WHEN origin.flow_name LIKE '%.gold.%' THEN 'Gold'
    ELSE 'Otro'
  END AS capa,
  SUM(COALESCE(CAST(details:flow_progress:metrics:num_output_rows AS BIGINT), 0)) AS cantidad_registros
FROM fintech_finpay.observability.event_logs
WHERE event_type = 'flow_progress'
GROUP BY 1
HAVING capa != 'Otro'
ORDER BY 1


## 2 · Registros fallidos por expectativas de calidad

In [ ]:
SELECT
  origin.flow_name AS tabla,
  SUM(CAST(details:flow_progress:data_quality:dropped_records AS BIGINT)) AS cantidad_rechazados
FROM fintech_finpay.observability.event_logs
WHERE event_type = 'flow_progress'
  AND CAST(details:flow_progress:data_quality:dropped_records AS BIGINT) >= 0 -- Quitamos el >0 para que veas los 0 si no hay errores
GROUP BY 1
ORDER BY cantidad_rechazados DESC;


## 3 · Tendencia diaria de filas procesadas por capa

In [ ]:
SELECT
  DATE(timestamp) AS fecha,
  CASE
    WHEN origin.flow_name LIKE '%.bronze.%' THEN 'Bronze'
    WHEN origin.flow_name LIKE '%.silver.%' THEN 'Silver'
    WHEN origin.flow_name LIKE '%.gold.%' THEN 'Gold'
    ELSE 'Otro'
  END AS capa,
  SUM(COALESCE(CAST(details:flow_progress:metrics:num_output_rows AS BIGINT), 0)) AS filas_procesadas
FROM fintech_finpay.observability.event_logs
WHERE event_type = 'flow_progress'
  AND origin.flow_name IS NOT NULL -- Quitamos logs que no son de tablas
GROUP BY 1, 2
HAVING capa != 'Otro' -- Solo mostramos las capas del Medallion
ORDER BY 1 DESC, 2;


## 4 · Duración de ejecuciones del pipeline

In [ ]:
SELECT
  origin.flow_name AS tabla,
  CASE
    WHEN origin.flow_name LIKE '%.bronze.%' THEN 'Bronze'
    WHEN origin.flow_name LIKE '%.silver.%' THEN 'Silver'
    WHEN origin.flow_name LIKE '%.gold.%' THEN 'Gold'
    ELSE 'Otro'
  END AS capa,
  SUM(CAST(details:flow_progress:metrics:num_output_rows AS BIGINT)) AS cantidad_registros
FROM fintech_finpay.observability.event_logs
WHERE event_type = 'flow_progress'
  AND details:flow_progress:metrics:num_output_rows IS NOT NULL
GROUP BY 1, 2
ORDER BY cantidad_registros DESC
LIMIT 100;


## 5 · Resumen de cuarentena por motivo de rechazo

In [ ]:
SELECT
  rejection_reason AS motivo_rechazo,
  source_name AS fuente,
  COUNT(*) AS cantidad_registros,
  MAX(processed_at) AS ultimo_incidente
FROM fintech_finpay.silver.quarantine
GROUP BY 1, 2
ORDER BY cantidad_registros DESC;
